<h2>Impact (v0.64) - 'Impact (v0.64).ipynb'</h2>

<u><h4>Context</h4></u>

The aim of the 'Impact' notebooks are to explore economic impacts resulsting from the FEM outputs in the lead-up to Autumn 2026, taking foward outputs from the associated 'Trade' notebooks.

<u><h4>Version Documentation and Future Work</h4></u>

'v0.2' cleaned up the analysis code by introducing the cost intensity estimation functions that were previously in the 'inventories' notebooks. For example, the 'get_usd_per_tnm' function is included here and applied to voyages identified as relevant to each individual trade record. The NZF costs associated with each trade record are now evaluated like this. 'v0.3' introduced the exporter- and importer-side trade portfolios at an HS-digit 4 level of resolution. With this implemented, the matching algorithm is now be being evaluated using the cargo weight field rather than value, and the process will be being conducted at the HS-digit 4 level of resolution. Since we are now confident regarding the use of the Global Shipping Watch dataset for the purposes of matching HS codes with facilitating vessel types, we should then be in good shape to take the analysis forward using the results produced in this notebook. 'v0.3' also subsumed the 'inventories' notebooks. In 'v0.4' sought to include transport costs data from UNCTAD's Trade-and-Transport database. However, there are now concerns regarding this dataset as it is importer-side only, and therefore has odd looking exporter-side portfolios. Instead, it makes sense for us to utilise trade portfolios derived from UN Comtrade, where these are submitted with disaggregation by Transport Mode. Looking at the 16 countries of interest across African and Caribbean economies we're interested in the summer of 2026, 13 of these have a trade portfolio with MoT Breakdown available across at least one year within the last decade. Therefore this approach makes more sense, and 'v0.6' was amended to work with Comtrade import and export trade portfolios downloaded from the platform. 'v0.61' made use of the updated 'trade' notebooks that derive a new weight field using Value-to-Weight and Weight-to-Value ratios observed in the CEPII BACI datasets across 2017-23. In the case of Trinidad and Tobaogo, import volumes were around 65% higher under the WtV approach which corresponded to a 39% greater 2041 BAU impact from FEM costs, but this was slightly less for exports where the weighted WtV was 9% more but 2041 FEM costs under BAU were around 10% less. A new 'composite' weight field has been added in Trade (v0.61) which only introduces the VtW-derived values where 'netWgt' is observed to be zero, however this resulted in only a tiny increase in recorded weight of around 1% for Angola, whereas the full VtW-derived field was ~50% greater. 

'v0.62' makes use of a new 'transport cost' field derived using ad-valorem transport cost rates identified from UNCTAD's Trade-and-Transport database. With these, we can now introduce another results disaggregation that presents NZF costs relative to expected transport costs, which is then fully in-line with how impacts are expressed in UNCTAD's CIA Task 3 work. 'v0.63' includes an assessment of 'Cost Intensity' matching process (similar to CIF-FOB matching), plus metrics on 'fac_vessel_types' and 'average sea-distance' (introduced in Trade v0.63). 'v0.64' has been modified throughout to account for a new scenario, namely the 'Japan' proposal, plus to remove the 'Low-then-High' scenario. In terms of the Impact notebook, this remains the same as 'v0.63'.

In [1]:
# Import packages
import os
import ast
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def voyages_analysis(voys):
    # Define a function to analyses key statistics from a given set of voyages, 'voys'.
    print("Number of Voyages found: {0}".format(voys.shape[0]))
    print("Total size of the fleet: {0} vessels".format(len(voys.imo.unique())))
    print("{0} vessel types contained in the dataset: {1}".format(len(voys.vessel_type.unique()), voys.vessel_type.unique()))
    print("Total transport work capacity: {0} million deadweight-tonne nautical miles".format(np.round(voys.dwtnm.sum() / 1e6, 1)))
    print("Total cargo transported: {0} tonnes".format(np.round(voys.tonnes.sum(), 1)))
    print("Total transport work: {0} million tonne nautical miles".format(np.round(voys.tnm.sum() / 1e6, 1)))
    print("Average build year of the fleet: {0}".format(int(np.round(voys.year_of_built.mean(), 0))))
    print("Total Energy Demands: {0} EJ".format(np.round(voys.energy_tj.sum(), 1)))
    print("Total CO2 produced: {0} tonnes".format(np.round(voys.co2_t.sum(), 1)))

def get_usd_per_dwtnm(voys, field_name):
    # Derive the deadweight-average of the cost intensity (per unit of cargo-carrying capacity) for the field 'field_name' in an input selection of voyages, 'voys'.
    return (voys[field_name].values * voys.deadweight.values).sum() / voys.deadweight.sum()
    
def get_usd_per_tnm(voys, field_name):
    # Derive the deadweight-average of the cost intensity (per unit of cargo-carried) for the field 'field_name' in an input selection of voyages, 'voys'.
    return (voys[field_name].values / voys.class_util.values * voys.tonnes.values).sum() / voys.tonnes.sum()

def download_link(df, title = "Download CSV file", filename = "data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

version = "v0.64"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
voys_fem_dir = "/Users/apple/repos/datasets/ucl/fleet evolution model/2026 annex vi amendments (v0.2)/2. NZF Impact Costs for 2018 IMO IV Voyages/voys_fem_v0.2.csv"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/impact/"
res_mlc = False

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "404"
iso_country = "Kenya"
year = 2020
import_portfolio = "h5_ag6_by_sea_M.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h5_ag6_by_sea_DX.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 100.660*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."
os.makedirs(output_dir+"{0}/results/".format(iso_country)) if not os.path.exists(output_dir+"{0}/results/".format(iso_country)) else "Output results folder already exists."

## 1.2 Voyages with FEM Results

In [3]:
voys_fem_d = {
    "source_iso":str, "source_iso_code":str, "source_iso_3":str, "source_iso_2":str, 
    "dest_iso_code":str, "dest_iso_code":str, "dest_iso_3":str, "dest_iso_2":str, 
    "source_region_code":str, "source_sub_region_code":str, "dest_region_code":str, "dest_sub_region_code":str, 
    "source_ldc":str, "source_sids":str, "dest_ldc":str, "dest_sids":str
}

voys_fem = pd.read_csv(voys_fem_dir, dtype=voys_fem_d)
voys_fem["dwtnm"] = voys_fem.deadweight * voys_fem.dist_travelled_ais
voys_fem["tonnes"] = voys_fem.deadweight * voys_fem.class_util
voys_fem["tnm"] = voys_fem.tonnes * voys_fem.dist_travelled_ais
voys_fem.loc[voys_fem.source_country == "Namibia", "source_iso_2"] = "NA"
voys_fem.loc[voys_fem.dest_country == "Namibia", "dest_iso_2"] = "NA"

countries_set = pd.Series(list(set(list(voys_fem.source_country.unique()) + list(voys_fem.dest_country.unique())))).sort_values()
print("Number of countries present in dataset: {0}.".format(len(countries_set)))
print("Number of IMO numbers present in dataset: {0}.".format(len(voys_fem.imo.unique())))
print("Total CO2 contained within the dataset: {0} million tonnes.".format(np.round(voys_fem.co2_t.sum() / 1e6, 1)))
print("Vessel types present in dataset: \n\n{0}\n".format(pd.Series(voys_fem.vessel_type.unique()).sort_values().values))
#print("\nCountries in dataset: \n\n", countries_set.values, "\n")
voys_fem.head(2)

Number of countries present in dataset: 207.
Number of IMO numbers present in dataset: 26132.
Total CO2 contained within the dataset: 544.7 million tonnes.
Vessel types present in dataset: 

['Bulk carrier' 'Chemical tanker' 'Container' 'General cargo'
 'Liquefied gas tanker' 'Oil tanker' 'Refrigerated bulk' 'Vehicle']



,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,vessel_type,gross_tonnage,capacity_unit,deadweight,teu,gas_cap,passengers,cars,size_id,size_range,year_of_built,flag,engine_type,me_fuel,class_med_aer,class_util,bau35_usd_dwtnm_26,bau35_usd_dwtnm_29,bau35_usd_dwtnm_32,bau35_usd_dwtnm_35,bau35_usd_dwtnm_38,bau35_usd_dwtnm_41,bau35_usd_dwtnm_44,bau35_usd_dwtnm_47,bau35_usd_dwtnm_50,levy35_usd_dwtnm_26,levy35_usd_dwtnm_29,levy35_usd_dwtnm_32,levy35_usd_dwtnm_35,levy35_usd_dwtnm_38,levy35_usd_dwtnm_41,levy35_usd_dwtnm_44,levy35_usd_dwtnm_47,levy35_usd_dwtnm_50,asis35_usd_dwtnm_26,asis35_usd_dwtnm_29,asis35_usd_dwtnm_32,asis35_usd_dwtnm_35,asis35_usd_dwtnm_38,asis35_usd_dwtnm_41,asis35_usd_dwtnm_44,asis35_usd_dwtnm_47,asis35_usd_dwtnm_50,brazil35_usd_dwtnm_26,brazil35_usd_dwtnm_29,brazil35_usd_dwtnm_32,brazil35_usd_dwtnm_35,brazil35_usd_dwtnm_38,brazil35_usd_dwtnm_41,brazil35_usd_dwtnm_44,brazil35_usd_dwtnm_47,brazil35_usd_dwtnm_50,liscr35_usd_dwtnm_26,liscr35_usd_dwtnm_29,liscr35_usd_dwtnm_32,liscr35_usd_dwtnm_35,liscr35_usd_dwtnm_38,liscr35_usd_dwtnm_41,liscr35_usd_dwtnm_44,liscr35_usd_dwtnm_47,liscr35_usd_dwtnm_50,direct35_usd_dwtnm_26,direct35_usd_dwtnm_29,direct35_usd_dwtnm_32,direct35_usd_dwtnm_35,direct35_usd_dwtnm_38,direct35_usd_dwtnm_41,direct35_usd_dwtnm_44,direct35_usd_dwtnm_47,direct35_usd_dwtnm_50,dwtnm,bau35_usd_26,bau35_usd_29,bau35_usd_32,bau35_usd_35,bau35_usd_38,bau35_usd_41,bau35_usd_44,bau35_usd_47,bau35_usd_50,levy35_usd_26,levy35_usd_29,levy35_usd_32,levy35_usd_35,levy35_usd_38,levy35_usd_41,levy35_usd_44,levy35_usd_47,levy35_usd_50,asis35_usd_26,asis35_usd_29,asis35_usd_32,asis35_usd_35,asis35_usd_38,asis35_usd_41,asis35_usd_44,asis35_usd_47,asis35_usd_50,brazil35_usd_26,brazil35_usd_29,brazil35_usd_32,brazil35_usd_35,brazil35_usd_38,brazil35_usd_41,brazil35_usd_44,brazil35_usd_47,brazil35_usd_50,liscr35_usd_26,liscr35_usd_29,liscr35_usd_32,liscr35_usd_35,liscr35_usd_38,liscr35_usd_41,liscr35_usd_44,liscr35_usd_47,liscr35_usd_50,direct35_usd_26,direct35_usd_29,direct35_usd_32,direct35_usd_35,direct35_usd_38,direct35_usd_41,direct35_usd_44,direct35_usd_47,direct35_usd_50,tonnes,tnm
0,0,5034525,413781699,CNTAX,156,CHN,CN,China,Taixing,CNNKG,156,CHN,CN,China,Nanjing,2018-01-01 06:00:00,2018-01-01 19:00:00,13.0,1.0,14.0,62.568523,0.0,0.0,0.0,3.635661,0.338483,3.974144,0.0,0.0,0.0,0.0,0.0,0.0,0.155243,0.014453,11.655930,1.085176,12.741107,142,Asia,030,Eastern Asia,upp_mid_income,0,0,142,Asia,030,Eastern Asia,upp_mid_income,0,0,0.169696,Refrigerated bulk,3786.0,dwt,3881.0,NaN,NaN,NaN,NaN,2,2000-5999,1959.0,Panama,SSD,4,76.1,0.666667,0.03155,0.032676,0.032143,0.031676,0.031218,0.030769,0.03031,0.029878,0.029438,0.031576,0.052178,0.052414,0.05804,0.060001,0.059297,0.058435,0.058228,0.058319,0.031568,0.034644,0.036843,0.044726,0.049621,0.053882,0.057797,0.059065,0.057999,0.031568,0.032727,0.034529,0.040735,0.048178,0.055864,0.057447,0.058672,0.05942,0.031559,0.032699,0.033808,0.037977,0.040129,0.041837,0.043315,0.044697,0.045876,0.031559,0.033985,0.034896,0.036131,0.038297,0.039924,0.040606,0.041099,0.041463,242828.439338,7661.320668,7934.549945,7805.304846,7691.950531,7580.696290,7471.542086,7360.199865,7255.307537,7148.350725,7667.458038,12670.391275,12727.642802,14093.647297,14570.027623,14398.931835,14189.590704,14139.488873,14161.462401,7665.715352,8412.516422,8946.6

# 2. Let's consider Imports

## 2.1 Read-in Trade dataset

In [4]:
trade_im_d = {
    "reporterCode":"str", "partnerCode":"str", "partner2Code":"str", 
    "cmdCode":"str", "HS2":"str", "HS4":"str", "hs2":"str", "hs4":"str", "hs6":"str",
    "isic":"str", "sitc":"str", "gtap":"str", "cpc":"str"
}
if res_mlc == False:
    trade_im = pd.read_csv(input_dir+"trade/{0}/trade_im_dist_{1}.csv".format(iso_country, version), dtype=trade_im_d)
else:
    trade_im = pd.read_csv(input_dir+"trade/{0}/trade_im_mlc_{1}.csv".format(iso_country, version), dtype=trade_im_d)
trade_im['fac_vessel_types'] = [ast.literal_eval(x) for x in trade_im.fac_vessel_types.values]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(trade_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(trade_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(trade_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_im.vol_kg_vtw.sum() / 1e6, 1)))

print("\nDistance to market: {0} nm (min) to {1} nm (max), {2} nm (observed mean)".format(
    round(trade_im.sea_dist_nm.min(), 1),
    round(trade_im.sea_dist_nm.max(), 1),
    round(trade_im.sea_dist_nm.mean(), 1)
))
print("Average distance to market (volume-weighted): {0} nm".format(
    round((trade_im.vol_kg_vtw / 1000 * trade_im.sea_dist_nm).sum() / (trade_im.vol_kg_vtw.sum() / 1000), 1)
))
print("Average distance to market (value-weighted): {0} nm".format(
    round((trade_im.cifvalue / 1000 * trade_im.sea_dist_nm).sum() / (trade_im.cifvalue.sum() / 1000), 1)
))
trade_im.head(2)

Aggregate trade portfolio statistics for 'Kenya': 

Number of trade records present in dataset: 32895
Number of partner countries present in dataset: 151
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$12510.8m
Total weight within the dataset: 13178.9 million kilograms

Distance to market: 0.0 nm (min) to 12365.9 nm (max), 5738.1 nm (observed mean)
Average distance to market (volume-weighted): 4702.7 nm
Average distance to market (value-weighted): 5275.7 nm


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,HS6_desc,HS6,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,HS6_bilat_desc,HS2,HS2_desc,HS2_bilat_desc,HS4,HS4_desc,HS4_bilat_desc,hs2,hs4,hs6,hs6_description,eora_desc,isic,isic_desc,sitc,sitc_desc,gtap,gtap_desc,cpc,cpc_desc,usd_per_kg,kg_per_usd_cepii,_trade_im_vtw_hs4_m,kg_per_usd_netWgt,vol_kg_vtw,vol_kg_vtw_comp,fac_vessel_types,fac_vessel_types_str,sea_dist_km,sea_dist_nm,tnm
0,0,404,Kenya,699,India,000,World,Import,Sea,"551321 | Fabrics, woven; plai",551321,kg,32898.0,32898.0,183953.222,NaN,183953.222,"India | 551321 | Fabrics, woven; plai",55,55 | Man-made staple fibr,India | 55 | Man-made staple fibr,5513,5513 | Woven fabrics of syn,India | 5513 | Woven fabrics of syn,55,5513,551321,"Woven fabrics of polyester staple fibres, cont...",Textiles and Wearing Apparel,1711,Preparation and spinning of textile fibres; we...,65331,"Fabrics, woven, <85% of synthetic fibres, of a...",27,TEX - Textiles,2676,"Woven fabrics of man-made staple fibres, conta...",6.4903,0.1541,both,0.178839,28347.191510,32898.0,[Container],['Container'],4305.0,2675.002155,75828.798378
1,1,404,Kenya,826,United Kingdom,000,World,Import,Sea,"551321 | Fabrics, woven; plai",551321,kg,1775.0,1775.0,6181.099,NaN,6181.099,"United Kingdom | 551321 | Fabrics, woven; plai",55,55 | Man-made staple fibr,United Kingdom | 55 | Man-made staple fibr,5513,5513 | Woven fabrics of syn,United Kingdom | 5513 | Woven fabrics of syn,55,5513,551321,"Woven fabrics of polyester staple fibres, cont...",Textiles and Wearing Apparel,1711,Preparation and spinning of textile fibres; we...,65331,"Fabrics, woven, <85% of synthetic fibres, of a...",27,TEX - Textiles,2676,"Woven fabrics of man-made staple fibres, conta...",6.4903,0.1541,both,0.287166,952.507356,1775.0,[Container],['Container'],11940.0,7419.169740,7066.813752


## 2.2 Derive Internationally Arriving Voyages

In [5]:
# International Arrivals
int_arr_voys = voys_fem[(voys_fem.dest_country == iso_country) & (voys_fem.source_country != iso_country)].reset_index(drop=True)
int_arr_voys.to_csv(output_dir + "{0}/int_arr_voys_{1}.csv".format(iso_country, version), index=False)
int_dep_voys = voys_fem[(voys_fem.source_country == iso_country) & (voys_fem.dest_country != iso_country)].reset_index(drop=True)
int_dep_voys.to_csv(output_dir + "{0}/int_dep_voys_{1}.csv".format(iso_country, version), index=False)
voyages_analysis(int_arr_voys)
int_arr_voys.head(2)

Number of Voyages found: 1150
Total size of the fleet: 633 vessels
6 vessel types contained in the dataset: ['Bulk carrier' 'General cargo' 'Container' 'Vehicle' 'Chemical tanker'
 'Oil tanker']
Total transport work capacity: 131810.4 million deadweight-tonne nautical miles
Total cargo transported: 27069285.5 tonnes
Total transport work: 74852.8 million tonne nautical miles
Average build year of the fleet: 2006
Total Energy Demands: 11847.8 EJ
Total CO2 produced: 916965.6 tonnes


,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,vessel_type,gross_tonnage,capacity_unit,deadweight,teu,gas_cap,passengers,cars,size_id,size_range,year_of_built,flag,engine_type,me_fuel,class_med_aer,class_util,bau35_usd_dwtnm_26,bau35_usd_dwtnm_29,bau35_usd_dwtnm_32,bau35_usd_dwtnm_35,bau35_usd_dwtnm_38,bau35_usd_dwtnm_41,bau35_usd_dwtnm_44,bau35_usd_dwtnm_47,bau35_usd_dwtnm_50,levy35_usd_dwtnm_26,levy35_usd_dwtnm_29,levy35_usd_dwtnm_32,levy35_usd_dwtnm_35,levy35_usd_dwtnm_38,levy35_usd_dwtnm_41,levy35_usd_dwtnm_44,levy35_usd_dwtnm_47,levy35_usd_dwtnm_50,asis35_usd_dwtnm_26,asis35_usd_dwtnm_29,asis35_usd_dwtnm_32,asis35_usd_dwtnm_35,asis35_usd_dwtnm_38,asis35_usd_dwtnm_41,asis35_usd_dwtnm_44,asis35_usd_dwtnm_47,asis35_usd_dwtnm_50,brazil35_usd_dwtnm_26,brazil35_usd_dwtnm_29,brazil35_usd_dwtnm_32,brazil35_usd_dwtnm_35,brazil35_usd_dwtnm_38,brazil35_usd_dwtnm_41,brazil35_usd_dwtnm_44,brazil35_usd_dwtnm_47,brazil35_usd_dwtnm_50,liscr35_usd_dwtnm_26,liscr35_usd_dwtnm_29,liscr35_usd_dwtnm_32,liscr35_usd_dwtnm_35,liscr35_usd_dwtnm_38,liscr35_usd_dwtnm_41,liscr35_usd_dwtnm_44,liscr35_usd_dwtnm_47,liscr35_usd_dwtnm_50,direct35_usd_dwtnm_26,direct35_usd_dwtnm_29,direct35_usd_dwtnm_32,direct35_usd_dwtnm_35,direct35_usd_dwtnm_38,direct35_usd_dwtnm_41,direct35_usd_dwtnm_44,direct35_usd_dwtnm_47,direct35_usd_dwtnm_50,dwtnm,bau35_usd_26,bau35_usd_29,bau35_usd_32,bau35_usd_35,bau35_usd_38,bau35_usd_41,bau35_usd_44,bau35_usd_47,bau35_usd_50,levy35_usd_26,levy35_usd_29,levy35_usd_32,levy35_usd_35,levy35_usd_38,levy35_usd_41,levy35_usd_44,levy35_usd_47,levy35_usd_50,asis35_usd_26,asis35_usd_29,asis35_usd_32,asis35_usd_35,asis35_usd_38,asis35_usd_41,asis35_usd_44,asis35_usd_47,asis35_usd_50,brazil35_usd_26,brazil35_usd_29,brazil35_usd_32,brazil35_usd_35,brazil35_usd_38,brazil35_usd_41,brazil35_usd_44,brazil35_usd_47,brazil35_usd_50,liscr35_usd_26,liscr35_usd_29,liscr35_usd_32,liscr35_usd_35,liscr35_usd_38,liscr35_usd_41,liscr35_usd_44,liscr35_usd_47,liscr35_usd_50,direct35_usd_26,direct35_usd_29,direct35_usd_32,direct35_usd_35,direct35_usd_38,direct35_usd_41,direct35_usd_44,direct35_usd_47,direct35_usd_50,tonnes,tnm
0,14050,7929308,369813000,TTPTP,780,TTO,TT,Trinidad and Tobago,Pointe A Pierre,KEMBA,404,KEN,KE,Kenya,Mombasa,2018-08-10 23:00:00,2018-09-06 11:00:00,636.0,213.0,849.0,8223.318527,709.827615,16.809613,726.637228,0.0000,0.0000,0.000,0.0,0.0,0.0,0.0,0.0,0.0,28.535070,0.675746,2210.403192,52.345134,2262.748327,019,Americas,419,Latin America and the Caribbean,high_income,0,1,002,Africa,202,Sub-Saharan Africa,low_mid_income,0,0,29.210817,Bulk carrier,22518.0,dwt,37244.0,NaN,NaN,NaN,NaN,3,35000-59999,1980.0,United States Of America,SSD,1,5.4,0.595506,0.003211,0.003315,0.003266,0.003223,0.003180,0.003139,0.003097,0.003057,0.003016,0.003206,0.005112,0.004962,0.005396,0.006129,0.006151,0.005833,0.005371,0.005094,0.003207,0.003469,0.003571,0.004255,0.004816,0.005923,0.005732,0.005588,0.005139,0.003207,0.003294,0.003436,0.003963,0.004598,0.006111,0.005994,0.006036,0.005850,0.003209,0.003304,0.003395,0.003761,0.003922,0.004434,0.005065,0.004957,0.004987,0.003209,0.003422,0.003494,0.003595,0.003778,0.003910,0.003957,0.003986,0.004003,3.062693e+08,983411.881982,1.015202e+06,1.000165e+06,986975.684199,974031.189123,961331.034538,948376.302898,936172.020121,923727.533357,981939.887153,1.565545e+06,1.51

## 2.3 Evaluate NZF Costs

In [6]:
years=["26", "29", "32", "35", "38", "41", "44", "47", "50"]
vess_types_not_found_in_inventories, vess_types_not_found_in_global_voys, hs_codes_not_mapped_to_HS, cost_int_match_desc, cost_int_match_desc_bool = [], [], [], [], False

for year in years:

    print("Processing game year 20{0}".format(year))

    # Setup columns for policy costs per bilateral goods trade.
    bau_usd, levy_usd, asis_usd, brazil_usd, liscr_usd, direct_usd = [], [], [], [], [], []
    
    for trade_idx, trade in trade_im.iterrows():

        # initiate selected voyages table as empty dataframe to begin
        int_arr_voys_vess = pd.DataFrame()

        # 3. If no facilitating vessel types were picked up for the HS code, take from full primary int. inv. voyages dataset, otherwise move on...
        if 'all' in trade.fac_vessel_types:
            hs_codes_not_mapped_to_HS.append(trade.HS4_desc)
            int_arr_voys_vess = int_arr_voys
            cost_int_match_desc.append("3. Primary Int. Inv. without Vess Type") if cost_int_match_desc_bool == False else None

        else:
            # 1. If possible, derive subset of voyages associated with the unique vessel type.
            int_arr_voys_vess = int_arr_voys[(int_arr_voys.vessel_type.isin(trade.fac_vessel_types))]    
            
            if int_arr_voys_vess.shape[0] > 0:
                cost_int_match_desc.append("1. Primary Int. Inv. with Vess Type") if cost_int_match_desc_bool == False else None
                
            else:
                # 2. Finally, if no voyages are found as the vessel type doesn't appear in the country's shipping inventories, also take from full voyages dataset filtered for those vessel types instead.
                int_arr_voys_vess = voys_fem[(voys_fem.vessel_type.isin(trade.fac_vessel_types))]

                if int_arr_voys_vess.shape[0] > 0:
                    cost_int_match_desc.append("2. Global Inv. with Vess Type") if cost_int_match_desc_bool == False else None
                    [vess_types_not_found_in_inventories.append(trade.fac_vessel_types[i]) for i in range(len(trade.fac_vessel_types))]

                else:
                    # 4. Use full global voyage dataset as a back-up
                    int_arr_voys_vess = voys_fem
                    cost_int_match_desc.append("4. Global Inv. without Vess Type.") if cost_int_match_desc_bool == False else None
                    [vess_types_not_found_in_global_voys.append(trade.fac_vessel_types[i]) for i in range(len(trade.fac_vessel_types))]

        # Add cost intensities associated with the subset of voyages by vessel type, assuming 50% of NZF costs are borne by the importer.
        bau_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "bau35_usd_dwtnm_"+year))
        levy_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "levy35_usd_dwtnm_"+year))
        asis_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "asis35_usd_dwtnm_"+year))
        brazil_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "brazil35_usd_dwtnm_"+year))
        liscr_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "liscr35_usd_dwtnm_"+year))
        direct_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "direct35_usd_dwtnm_"+year))
    
    if cost_int_match_desc_bool == False:
        trade_im["cost_int_match_desc"] = cost_int_match_desc
    cost_int_match_desc_bool = True
    
    trade_im["bau_usd_"+year] = bau_usd
    trade_im["levy_usd_"+year] = levy_usd
    trade_im["asis_usd_"+year] = asis_usd
    trade_im["brazil_usd_"+year] = brazil_usd
    trade_im["liscr_usd_"+year] = liscr_usd
    trade_im["direct_usd_"+year] = direct_usd

print("Inventory voyages table used for the following vessel types: \n\n{0}\n".format(pd.Series(vess_types_not_found_in_inventories).unique()))
print("Global voyages table used for the following vessel types found: \n\n{0}\n".format(pd.Series(vess_types_not_found_in_global_voys).unique()))
print("Global voyages table used for the following commodities as no 'facilitating vessel types' were identified: \n\n{0}\n".format(pd.Series(hs_codes_not_mapped_to_HS).unique()))

print("\nNumber of trade records present in dataset: {0}".format(trade_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_im.partnerCode.unique())))
print("\nTotal value of trade: US${0}m".format(np.round(trade_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_im.vol_kg_vtw.sum() / 1e6, 1)))

print("\nDistance to market: {0} nm (min) to {1} nm (max), {2} nm (observed mean)".format(
    round(trade_im.sea_dist_nm.min(), 1),
    round(trade_im.sea_dist_nm.max(), 1),
    round(trade_im.sea_dist_nm.mean(), 1)
))
print("Average distance to market (volume-weighted): {0} nm".format(
    round((trade_im.vol_kg_vtw / 1000 * trade_im.sea_dist_nm).sum() / (trade_im.vol_kg_vtw.sum() / 1000), 1)
))
print("Average distance to market (value-weighted): {0} nm".format(
    round((trade_im.cifvalue / 1000 * trade_im.sea_dist_nm).sum() / (trade_im.cifvalue.sum() / 1000), 1)
))

print("\n\nTotal FEM costs in 2041...\n")
print("BAU: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.bau_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.bau_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("Levy: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.levy_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.levy_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("'as-is': US${0}m ({1}% ad-val)".format(
    np.round(trade_im.asis_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.asis_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("Brazil: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.brazil_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.brazil_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("LISCR: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.liscr_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.liscr_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("Direct: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.direct_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.direct_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))

trade_im_agg = trade_im.groupby(["cost_int_match_desc"]).agg({"vol_kg_vtw":"sum", "cifvalue":"sum"}).reset_index().sort_values(by="vol_kg_vtw", ascending=False).reset_index(drop=True)
trade_im_agg["vol_kg_vtw_pct"] = round(100 * trade_im_agg.vol_kg_vtw / trade_im_agg.vol_kg_vtw.sum(), 1)
trade_im_agg["cifvalue_pct"] = round(100 * trade_im_agg.cifvalue / trade_im_agg.cifvalue.sum(), 1)
print("Most prominent 'cost intensity match' types...\n\n{0}\n".format(
    trade_im_agg[["cost_int_match_desc", "vol_kg_vtw_pct", "cifvalue_pct"]].head(5)
))
trade_im_agg_2 = trade_im.groupby(["fac_vessel_types_str", "cost_int_match_desc"]).agg({"vol_kg_vtw":"sum", "cifvalue":"sum"}).reset_index().sort_values(by="vol_kg_vtw", ascending=False).reset_index(drop=True)
trade_im_agg_2["vol_kg_vtw_pct"] = round(100 * trade_im_agg_2.vol_kg_vtw / trade_im_agg_2.vol_kg_vtw.sum(), 1)
trade_im_agg_2["cifvalue_pct"] = round(100 * trade_im_agg_2.cifvalue / trade_im_agg_2.cifvalue.sum(), 1)
print("Most prominent 'facilitating vessel types' and their 'cost intensity match' types...")
display(trade_im_agg_2[["fac_vessel_types_str", "cost_int_match_desc", "vol_kg_vtw_pct", "cifvalue_pct"]].head(5))

trade_im_na = trade_im[(trade_im[["vol_kg_vtw", "primaryValue", "HS2", "HS6", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (trade_im.fac_vessel_types.str.len() == 0)]
trade_im = trade_im[(~trade_im[["vol_kg_vtw", "primaryValue", "HS2", "HS6", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (~trade_im.fac_vessel_types.str.len() == 0)]
print("Rows with NA: ${0} ({1}%)\n".format(trade_im_na.primaryValue.sum(), round(100 * trade_im_na.primaryValue.sum() / trade_im.primaryValue.sum(), 1)))

trade_im.to_csv(output_dir + "{0}/impact_im_{1}.csv".format(iso_country, version), index=False)
trade_im.head(2)

Processing game year 2026
Processing game year 2029
Processing game year 2032
Processing game year 2035
Processing game year 2038
Processing game year 2041
Processing game year 2044
Processing game year 2047
Processing game year 2050
Inventory voyages table used for the following vessel types: 

['Liquefied gas tanker']

Global voyages table used for the following vessel types found: 

['Cruise' 'Other liquids tankers' 'Ro-Ro']

Global voyages table used for the following commodities as no 'facilitating vessel types' were identified: 

['5516 | Woven fabrics of art' '2914 | Ketones and quinones'
 '5703 | Carpets and other te' '5802 | Fabrics; terry towel'
 '2920 | Esters of other inor' '8462 | Machine-tools (inclu'
 '2922 | Oxygen-function amin' '5911 | Textile products and'
 '6005 | Fabrics; warp knit (' '0307 | Molluscs; whether in'
 '2931 | Other organo-inorgan' '0403 | Yogurt; buttermilk, '
 '6006 | Fabrics; knitted or ' '0604 | Foliage, branches an'
 '2932 | Heterocyclic compoun' 

,fac_vessel_types_str,cost_int_match_desc,vol_kg_vtw_pct,cifvalue_pct
0,['Container'],1. Primary Int. Inv. with Vess Type,33.4,48.1
1,['Bulk carrier'],1. Primary Int. Inv. with Vess Type,21.7,7.4
2,['Chemical tanker'],1. Primary Int. Inv. with Vess Type,15.7,12.0
3,['Oil tanker'],1. Primary Int. Inv. with Vess Type,15.1,10.0
4,"['Container', 'Bulk carrier']",1. Primary Int. Inv. with Vess Type,3.2,1.8


Rows with NA: $0.0 (0.0%)



,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,HS6_desc,HS6,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,HS6_bilat_desc,HS2,HS2_desc,HS2_bilat_desc,HS4,HS4_desc,HS4_bilat_desc,hs2,hs4,hs6,hs6_description,eora_desc,isic,isic_desc,sitc,sitc_desc,gtap,gtap_desc,cpc,cpc_desc,usd_per_kg,kg_per_usd_cepii,_trade_im_vtw_hs4_m,kg_per_usd_netWgt,vol_kg_vtw,vol_kg_vtw_comp,fac_vessel_types,fac_vessel_types_str,sea_dist_km,sea_dist_nm,tnm,cost_int_match_desc,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50
0,0,404,Kenya,699,India,000,World,Import,Sea,"551321 | Fabrics, woven; plai",551321,kg,32898.0,32898.0,183953.222,NaN,183953.222,"India | 551321 | Fabrics, woven; plai",55,55 | Man-made staple fibr,India | 55 | Man-made staple fibr,5513,5513 | Woven fabrics of syn,India | 5513 | Woven fabrics of syn,55,5513,551321,"Woven fabrics of polyester staple fibres, cont...",Textiles and Wearing Apparel,1711,Preparation and spinning of textile fibres; we...,65331,"Fabrics, woven, <85% of synthetic fibres, of a...",27,TEX - Textiles,2676,"Woven fabrics of man-made staple fibres, conta...",6.4903,0.1541,both,0.178839,28347.191510,32898.0,[Container],['Container'],4305.0,2675.002155,75828.798378,1. Primary Int. Inv. with Vess Type,257.572910,256.624104,256.694835,256.694835,257.294932,257.294932,270.641375,509.269469,298.495209,275.500922,276.563552,284.470778,264.437671,484.450716,311.402575,286.553171,279.943997,294.074767,259.001720,537.003253,391.019959,340.430261,314.539330,307.334343,253.642223,569.658752,457.007089,426.391727,333.566018,330.569120,248.381239,581.580095,548.994422,558.283666,360.779346,347.728191,243.019194,571.261722,546.663364,580.468874,436.574899,354.336567,237.971708,505.714846,521.563876,545.687883,404.674383,358.668978,232.841733,455.691395,466.078941,538.860616,445.407651,361.486269
1,1,404,Kenya,826,United Kingdom,000,World,Import,Sea,"551321 | Fabrics, woven; plai",551321,kg,1775.0,1775.0,6181.099,NaN,6181.099,"United Kingdom | 551321 | Fabrics, woven; plai",55,55 | Man-made staple fibr,United Kingdom | 55 | Man-made staple fibr,5513,5513 | Woven fabrics of syn,United Kingdom | 5513 | Woven fabrics of syn,55,5513,551321,"Woven fabrics of polyester staple fibres, cont...",Textiles and Wearing Apparel,1711,Preparation and spinning of textile fibres; we...,65331,"Fabrics, woven, <85% of synthetic fibres, of a...",27,TEX - Textiles,2676,"Woven fabrics of man-made staple fibres, conta...",6.4903,0.1541,both,0.287166,952.507356,1775.0,[Container],['Container'],11940.0,7419.169740,7066.813752,1. Primary Int. Inv. with Vess Type,24.004334,23.915910,23.922502,23.922502,23.978428,23.978428,25.222241,47.461025,27.818060,25.675123,25.774154,26.511062,24.644090,45.148058,29.020953,26.705130,26.089192,27.406100,24.137491,50.045656,36.440841,31.726169,29.313281,28.641817,23.638016,53.088963,42.590468,39.737290,31.086460,30.807166,23.147722,54.199965,51.163165,52.028870,33.622588,32.406294,22.648010,53.238351,50.945924,54.096405,40.686303,33.022158,22.177613,47.129754,48.606794,50.855014,37.713356,33.425914,21.699528,42.467852,43.435913,50.218752,41.509466,33.688469


## 2.4 Create Results Tables

### 2.4.1 Total Costs per Scenario

In [7]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        np.round(trade_im["bau_usd_{0}".format(year)].sum(), 2),
        np.round(trade_im["levy_usd_{0}".format(year)].sum(), 2),
        np.round(trade_im["asis_usd_{0}".format(year)].sum(), 2),
        np.round(trade_im["brazil_usd_{0}".format(year)].sum(), 2),
        np.round(trade_im["liscr_usd_{0}".format(year)].sum(), 2),
        np.round(trade_im["direct_usd_{0}".format(year)].sum(), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)

if res_mlc == True:
    print("\nTotal FEM costs by scenario (% ciffob)...\n")
    d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * trade_im["bau_usd_{0}".format(year)].sum() / trade_im.TandT_ciffob.sum(), 2),
            round(100 * trade_im["levy_usd_{0}".format(year)].sum() / trade_im.TandT_ciffob.sum(), 2),
            round(100 * trade_im["asis_usd_{0}".format(year)].sum() / trade_im.TandT_ciffob.sum(), 2),
            round(100 * trade_im["brazil_usd_{0}".format(year)].sum() / trade_im.TandT_ciffob.sum(), 2),
            round(100 * trade_im["liscr_usd_{0}".format(year)].sum() / trade_im.TandT_ciffob.sum(), 2),
            round(100 * trade_im["direct_usd_{0}".format(year)].sum() / trade_im.TandT_ciffob.sum(), 2),
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_im_total_costs_pct_ciffob_{1}.csv".format(iso_country, version), index=False)
    display(df)

print("\nTotal FEM costs by scenario (% cif)...\n")
d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_im["bau_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 2),
        round(100 * trade_im["levy_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 2),
        round(100 * trade_im["asis_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 2),
        round(100 * trade_im["brazil_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 2),
        round(100 * trade_im["liscr_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 2),
        round(100 * trade_im["direct_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)

if res_mlc == True:
    print("\nTotal FEM costs by scenario (% fob)...\n")
    d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * trade_im["bau_usd_{0}".format(year)].sum() / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * trade_im["levy_usd_{0}".format(year)].sum() / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * trade_im["asis_usd_{0}".format(year)].sum() / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * trade_im["brazil_usd_{0}".format(year)].sum() / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * trade_im["liscr_usd_{0}".format(year)].sum() / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * trade_im["direct_usd_{0}".format(year)].sum() / trade_im.TandT_fobvalue.sum(), 2)
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_im_total_costs_pct_fob_{1}.csv".format(iso_country, version), index=False)
    display(df)

print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_im["bau_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["levy_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["asis_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["brazil_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["liscr_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["direct_usd_{0}".format(year)].sum() / gdp, 3),
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_total_costs_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,2.177744e+08,2.135531e+08,2.099400e+08,2.063847e+08,2.028869e+08,1.992731e+08,1.958866e+08,1.923517e+08
1,Levy,3.746315e+08,3.602847e+08,3.978084e+08,4.319141e+08,4.355993e+08,4.197401e+08,3.808945e+08,3.480535e+08
2,'as-is',2.344367e+08,2.450224e+08,2.964432e+08,3.456917e+08,4.128485e+08,4.076326e+08,3.895500e+08,3.554393e+08
3,Brazil,2.193486e+08,2.282275e+08,2.668994e+08,3.253498e+08,4.244061e+08,4.311129e+08,4.185283e+08,4.067266e+08
4,LISCR,2.201134e+08,2.236862e+08,2.494611e+08,2.625314e+08,2.830900e+08,3.368292e+08,3.238113e+08,3.409305e+08
5,Direct,2.269326e+08,2.331080e+08,2.421052e+08,2.580109e+08,2.697652e+08,2.741899e+08,2.775932e+08,2.797465e+08



Total FEM costs by scenario (% cif)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,1.74,1.71,1.68,1.65,1.62,1.59,1.57,1.54
1,Levy,2.99,2.88,3.18,3.45,3.48,3.36,3.04,2.78
2,'as-is',1.87,1.96,2.37,2.76,3.30,3.26,3.11,2.84
3,Brazil,1.75,1.82,2.13,2.60,3.39,3.45,3.35,3.25
4,LISCR,1.76,1.79,1.99,2.10,2.26,2.69,2.59,2.73
5,Direct,1.81,1.86,1.94,2.06,2.16,2.19,2.22,2.24



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,0.216,0.212,0.209,0.205,0.202,0.198,0.195,0.191
1,Levy,0.372,0.358,0.395,0.429,0.433,0.417,0.378,0.346
2,'as-is',0.233,0.243,0.294,0.343,0.410,0.405,0.387,0.353
3,Brazil,0.218,0.227,0.265,0.323,0.422,0.428,0.416,0.404
4,LISCR,0.219,0.222,0.248,0.261,0.281,0.335,0.322,0.339
5,Direct,0.225,0.232,0.241,0.256,0.268,0.272,0.276,0.278


### 2.4.2 Costs per Scenario relative to BAU

In [8]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        np.round((trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_im["direct_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_costs_to_bau_usd_bau_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal change in FEM costs by scenario (% bau)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        round((100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_im["direct_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 2)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_costs_to_bau_pct_bau_{1}.csv".format(iso_country, version), index=False)
display(df)

if res_mlc == True:
    print("\nTotal FEM costs by scenario (% ciffob)...\n")
    d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_ciffob.sum(), 2),
            round(100 * (trade_im["direct_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_ciffob.sum(), 2),
            round(100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_ciffob.sum(), 2),
            round(100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_ciffob.sum(), 2),
            round(100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_ciffob.sum(), 2)
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_im_costs_to_bau_pct_ciffob_{1}.csv".format(iso_country, version), index=False)
    display(df)

print("\nTotal FEM costs by scenario (% cif)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 2),
        round(100 * (trade_im["direct_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 2),
        round(100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 2),
        round(100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 2),
        round(100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 2)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_costs_to_bau_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)

if res_mlc == True:
    print("\nTotal FEM costs by scenario (% fob)...\n")
    d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * (trade_im["direct_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_fobvalue.sum(), 2)
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_im_costs_to_bau_pct_fob_{1}.csv".format(iso_country, version), index=False)
    display(df)
    
print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["direct_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3)
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_costs_to_bau_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,1.568571e+08,1.467315e+08,1.878684e+08,2.255295e+08,2.327123e+08,2.204671e+08,1.850078e+08,1.557018e+08
1,'as-is',1.666232e+07,3.146925e+07,8.650322e+07,1.393070e+08,2.099616e+08,2.083595e+08,1.936633e+08,1.630876e+08
2,Brazil,1.574226e+06,1.467436e+07,5.695934e+07,1.189652e+08,2.215192e+08,2.318398e+08,2.226416e+08,2.143748e+08
3,LISCR,2.339021e+06,1.013309e+07,3.952103e+07,5.614670e+07,8.020309e+07,1.375561e+08,1.279247e+08,1.485788e+08
4,Direct,9.158221e+06,1.955481e+07,3.216515e+07,5.162629e+07,6.687822e+07,7.491682e+07,8.170660e+07,8.739480e+07



Total change in FEM costs by scenario (% bau)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,72.03,68.71,89.49,109.28,114.70,110.64,94.45,80.95
1,'as-is',4.21,9.16,15.32,25.01,32.96,37.60,41.71,45.43
2,Brazil,7.65,14.74,41.20,67.50,103.49,104.56,98.86,84.79
3,LISCR,0.72,6.87,27.13,57.64,109.18,116.34,113.66,111.45
4,Direct,1.07,4.74,18.82,27.20,39.53,69.03,65.31,77.24



Total FEM costs by scenario (% cif)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,1.25,1.17,1.50,1.80,1.86,1.76,1.48,1.24
1,'as-is',0.07,0.16,0.26,0.41,0.53,0.60,0.65,0.70
2,Brazil,0.13,0.25,0.69,1.11,1.68,1.67,1.55,1.30
3,LISCR,0.01,0.12,0.46,0.95,1.77,1.85,1.78,1.71
4,Direct,0.02,0.08,0.32,0.45,0.64,1.10,1.02,1.19



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,0.156,0.146,0.187,0.224,0.231,0.219,0.184,0.155
1,'as-is',0.009,0.019,0.032,0.051,0.066,0.074,0.081,0.087
2,Brazil,0.017,0.031,0.086,0.138,0.209,0.207,0.192,0.162
3,LISCR,0.002,0.015,0.057,0.118,0.220,0.230,0.221,0.213
4,Direct,0.002,0.010,0.039,0.056,0.080,0.137,0.127,0.148


# 3. Let's consider Exports

## 3.1 Read-in Trade dataset

In [9]:
trade_ex_d = {
    "reporterCode":"str", "partnerCode":"str", "partner2Code":"str", 
    "cmdCode":"str", "HS2":"str", "HS4":"str", "hs2":"str", "hs4":"str", "hs6":"str",
    "isic":"str", "sitc":"str", "gtap":"str", "cpc":"str"
}
if res_mlc == False:
    trade_ex = pd.read_csv(input_dir+"trade/{0}/trade_ex_dist_{1}.csv".format(iso_country, version), dtype=trade_ex_d)
else:
    trade_ex = pd.read_csv(input_dir+"trade/{0}/trade_ex_mlc_{1}.csv".format(iso_country, version), dtype=trade_ex_d)
trade_ex['fac_vessel_types'] = [ast.literal_eval(x) for x in trade_ex.fac_vessel_types.values]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(trade_ex.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_ex.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(trade_ex.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(trade_ex.fobvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_ex.vol_kg_vtw.sum() / 1e6, 1)))

print("\nDistance to market: {0} nm (min) to {1} nm (max), {2} nm (observed mean)".format(
    round(trade_ex.sea_dist_nm.min(), 1),
    round(trade_ex.sea_dist_nm.max(), 1),
    round(trade_ex.sea_dist_nm.mean(), 1)
))
print("Average distance to market (volume-weighted): {0} nm".format(
    round((trade_ex.vol_kg_vtw / 1000 * trade_ex.sea_dist_nm).sum() / (trade_ex.vol_kg_vtw.sum() / 1000), 1)
))
print("Average distance to market (value-weighted): {0} nm".format(
    round((trade_ex.fobvalue / 1000 * trade_ex.sea_dist_nm).sum() / (trade_ex.fobvalue.sum() / 1000), 1)
))
trade_ex.head(2)

Aggregate trade portfolio statistics for 'Kenya': 

Number of trade records present in dataset: 5357
Number of partner countries present in dataset: 143
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$2726.9m
Total weight within the dataset: 1589.1 million kilograms

Distance to market: 0.0 nm (min) to 12365.3 nm (max), 4082.5 nm (observed mean)
Average distance to market (volume-weighted): 4925.2 nm
Average distance to market (value-weighted): 5521.6 nm


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,HS6_desc,HS6,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,HS6_bilat_desc,HS2,HS2_desc,HS2_bilat_desc,HS4,HS4_desc,HS4_bilat_desc,hs2,hs4,hs6,hs6_description,eora_desc,isic,isic_desc,sitc,sitc_desc,gtap,gtap_desc,cpc,cpc_desc,usd_per_kg,kg_per_usd_cepii,_trade_ex_vtw_hs4_m,kg_per_usd_netWgt,vol_kg_vtw,vol_kg_vtw_comp,fac_vessel_types,fac_vessel_types_str,sea_dist_km,sea_dist_nm,tnm
0,0,404,Kenya,340,Honduras,000,World,Domestic Export,Sea,"551323 | Fabrics, woven; dyed",551323,kg,5182.0,5182.0,NaN,27151.845,27151.845,"Honduras | 551323 | Fabrics, woven; dyed",55,55 | Man-made staple fibr,Honduras | 55 | Man-made staple fibr,5513,5513 | Woven fabrics of syn,Honduras | 5513 | Woven fabrics of syn,55,5513,551323,"Dyed Woven fabrics of synthetic, polyester sta...",Textiles and Wearing Apparel,1711,Preparation and spinning of textile fibres; we...,65331,"Fabrics, woven, <85% of synthetic fibres, of a...",27,TEX - Textiles,2676,"Woven fabrics of man-made staple fibres, conta...",6.4903,0.1541,both,0.190853,4184.099315,5182.0,[Container],['Container'],17795.0,11057.296945,46264.828568
1,1,404,Kenya,826,United Kingdom,000,World,Domestic Export,Sea,"551441 | Fabrics, woven; plai",551441,kg,30.0,30.0,NaN,255.592,255.592,"United Kingdom | 551441 | Fabrics, woven; plai",55,55 | Man-made staple fibr,United Kingdom | 55 | Man-made staple fibr,5514,5514 | Woven fabrics of syn,United Kingdom | 5514 | Woven fabrics of syn,55,5514,551441,"Woven fabrics of polyester staple fibres, cont...",Textiles and Wearing Apparel,1711,Preparation and spinning of textile fibres; we...,65333,"Fabrics, woven, <85% of synthetic fibres, of a...",27,TEX - Textiles,2676,"Woven fabrics of man-made staple fibres, conta...",5.5692,0.1796,both,0.117375,45.904323,30.0,[Container],['Container'],11940.0,7419.169740,340.571966


## 3.2 Derive Internationally Departing Voyages

In [10]:
# International Departures
int_arr_voys = voys_fem[(voys_fem.dest_country == iso_country) & (voys_fem.source_country != iso_country)].reset_index(drop=True)
int_arr_voys.to_csv(output_dir + "{0}/int_arr_voys_{1}.csv".format(iso_country, version), index=False)
int_dep_voys = voys_fem[(voys_fem.source_country == iso_country) & (voys_fem.dest_country != iso_country)].reset_index(drop=True)
int_dep_voys.to_csv(output_dir + "{0}/int_dep_voys_{1}.csv".format(iso_country, version), index=False)
voyages_analysis(int_dep_voys)
int_dep_voys.head(2)

Number of Voyages found: 1161
Total size of the fleet: 642 vessels
6 vessel types contained in the dataset: ['Bulk carrier' 'General cargo' 'Container' 'Chemical tanker' 'Vehicle'
 'Oil tanker']
Total transport work capacity: 91332.8 million deadweight-tonne nautical miles
Total cargo transported: 27432629.1 tonnes
Total transport work: 50685.6 million tonne nautical miles
Average build year of the fleet: 2006
Total Energy Demands: 7844.3 EJ
Total CO2 produced: 606985.5 tonnes


,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,vessel_type,gross_tonnage,capacity_unit,deadweight,teu,gas_cap,passengers,cars,size_id,size_range,year_of_built,flag,engine_type,me_fuel,class_med_aer,class_util,bau35_usd_dwtnm_26,bau35_usd_dwtnm_29,bau35_usd_dwtnm_32,bau35_usd_dwtnm_35,bau35_usd_dwtnm_38,bau35_usd_dwtnm_41,bau35_usd_dwtnm_44,bau35_usd_dwtnm_47,bau35_usd_dwtnm_50,levy35_usd_dwtnm_26,levy35_usd_dwtnm_29,levy35_usd_dwtnm_32,levy35_usd_dwtnm_35,levy35_usd_dwtnm_38,levy35_usd_dwtnm_41,levy35_usd_dwtnm_44,levy35_usd_dwtnm_47,levy35_usd_dwtnm_50,asis35_usd_dwtnm_26,asis35_usd_dwtnm_29,asis35_usd_dwtnm_32,asis35_usd_dwtnm_35,asis35_usd_dwtnm_38,asis35_usd_dwtnm_41,asis35_usd_dwtnm_44,asis35_usd_dwtnm_47,asis35_usd_dwtnm_50,brazil35_usd_dwtnm_26,brazil35_usd_dwtnm_29,brazil35_usd_dwtnm_32,brazil35_usd_dwtnm_35,brazil35_usd_dwtnm_38,brazil35_usd_dwtnm_41,brazil35_usd_dwtnm_44,brazil35_usd_dwtnm_47,brazil35_usd_dwtnm_50,liscr35_usd_dwtnm_26,liscr35_usd_dwtnm_29,liscr35_usd_dwtnm_32,liscr35_usd_dwtnm_35,liscr35_usd_dwtnm_38,liscr35_usd_dwtnm_41,liscr35_usd_dwtnm_44,liscr35_usd_dwtnm_47,liscr35_usd_dwtnm_50,direct35_usd_dwtnm_26,direct35_usd_dwtnm_29,direct35_usd_dwtnm_32,direct35_usd_dwtnm_35,direct35_usd_dwtnm_38,direct35_usd_dwtnm_41,direct35_usd_dwtnm_44,direct35_usd_dwtnm_47,direct35_usd_dwtnm_50,dwtnm,bau35_usd_26,bau35_usd_29,bau35_usd_32,bau35_usd_35,bau35_usd_38,bau35_usd_41,bau35_usd_44,bau35_usd_47,bau35_usd_50,levy35_usd_26,levy35_usd_29,levy35_usd_32,levy35_usd_35,levy35_usd_38,levy35_usd_41,levy35_usd_44,levy35_usd_47,levy35_usd_50,asis35_usd_26,asis35_usd_29,asis35_usd_32,asis35_usd_35,asis35_usd_38,asis35_usd_41,asis35_usd_44,asis35_usd_47,asis35_usd_50,brazil35_usd_26,brazil35_usd_29,brazil35_usd_32,brazil35_usd_35,brazil35_usd_38,brazil35_usd_41,brazil35_usd_44,brazil35_usd_47,brazil35_usd_50,liscr35_usd_26,liscr35_usd_29,liscr35_usd_32,liscr35_usd_35,liscr35_usd_38,liscr35_usd_41,liscr35_usd_44,liscr35_usd_47,liscr35_usd_50,direct35_usd_26,direct35_usd_29,direct35_usd_32,direct35_usd_35,direct35_usd_38,direct35_usd_41,direct35_usd_44,direct35_usd_47,direct35_usd_50,tonnes,tnm
0,14051,7929308,369813000,KEMBA,404,KEN,KE,Kenya,Mombasa,DJJIB,262,DJI,DJ,Djibouti,Djibouti,2018-09-15 08:00:00,2018-09-22 05:00:00,165.0,95.0,260.0,2400.787234,268.014646,9.676742,277.691388,0.0000,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,10.774189,0.389005,834.597609,30.133375,864.730984,002,Africa,202,Sub-Saharan Africa,low_mid_income,0,0,002,Africa,202,Sub-Saharan Africa,low_mid_income,1,0,11.163194,Bulk carrier,22518.0,dwt,37244.0,NaN,NaN,NaN,NaN,3,35000-59999,1980.0,United States Of America,SSD,1,5.4,0.595506,0.003211,0.003315,0.003266,0.003223,0.003180,0.003139,0.003097,0.003057,0.003016,0.003206,0.005112,0.004962,0.005396,0.006129,0.006151,0.005833,0.005371,0.005094,0.003207,0.003469,0.003571,0.004255,0.004816,0.005923,0.005732,0.005588,0.005139,0.003207,0.003294,0.003436,0.003963,0.004598,0.006111,0.005994,0.006036,0.005850,0.003209,0.003304,0.003395,0.003761,0.003922,0.004434,0.005065,0.004957,0.004987,0.003209,0.003422,0.003494,0.003595,0.003778,0.003910,0.003957,0.003986,0.004003,8.941492e+07,287105.830092,296386.991626,291996.742322,288146.277551,284367.148915,280659.355171,276877.237983,273314.213373,269681.061519,286676.083101,4.570588e+05,443709.846835,4.824922e+05,5.479825

## 3.3 Evaluate NZF Costs

In [11]:
years=["26", "29", "32", "35", "38", "41", "44", "47", "50"]
vess_types_not_found_in_inventories, vess_types_not_found_in_global_voys, hs_codes_not_mapped_to_HS, cost_int_match_desc, cost_int_match_desc_bool = [], [], [], [], False

for year in years:

    print("Processing game year 20{0}".format(year))

    # Setup columns for policy costs per bilateral goods trade.
    bau_usd, levy_usd, asis_usd, brazil_usd, liscr_usd, direct_usd = [], [], [], [], [], []
    
    for trade_idx, trade in trade_ex.iterrows():

        # initiate selected voyages table as empty dataframe to begin
        int_dep_voys_vess = pd.DataFrame()

        # 3. If no facilitating vessel types were picked up for the HS code, take from full primary int. inv. voyages dataset, otherwise move on...
        if 'all' in trade.fac_vessel_types:
            hs_codes_not_mapped_to_HS.append(trade.HS4_desc)
            int_dep_voys_vess = int_dep_voys
            cost_int_match_desc.append("3. Primary Int. Inv. without Vess Type") if cost_int_match_desc_bool == False else None

        else:
            # 1. If possible, derive subset of voyages associated with the unique vessel type.
            int_dep_voys_vess = int_dep_voys[(int_dep_voys.vessel_type.isin(trade.fac_vessel_types))]    
            
            if int_dep_voys_vess.shape[0] > 0:
                cost_int_match_desc.append("1. Primary Int. Inv. with Vess Type") if cost_int_match_desc_bool == False else None
                
            else:
                # 2. Finally, if no voyages are found as the vessel type doesn't appear in the country's shipping inventories, also take from full voyages dataset filtered for those vessel types instead.
                int_dep_voys_vess = voys_fem[(voys_fem.vessel_type.isin(trade.fac_vessel_types))]
                
                if int_dep_voys_vess.shape[0] > 0:
                    cost_int_match_desc.append("2. Global Inv. with Vess Type") if cost_int_match_desc_bool == False else None
                    [vess_types_not_found_in_inventories.append(trade.fac_vessel_types[i]) for i in range(len(trade.fac_vessel_types))]

                else:
                    # 4. Use full global voyage dataset as a back-up
                    int_dep_voys_vess = voys_fem
                    cost_int_match_desc.append("4. Global Inv. without Vess Type.") if cost_int_match_desc_bool == False else None
                    [vess_types_not_found_in_global_voys.append(trade.fac_vessel_types[i]) for i in range(len(trade.fac_vessel_types))]
    
        # Add cost intensities associated with the subset of voyages by vessel type, assuming 50% of NZF costs are borne by the exporter.
        bau_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "bau35_usd_dwtnm_"+year))
        levy_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "levy35_usd_dwtnm_"+year))
        asis_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "asis35_usd_dwtnm_"+year))
        brazil_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "brazil35_usd_dwtnm_"+year))
        liscr_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "liscr35_usd_dwtnm_"+year))
        direct_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "direct35_usd_dwtnm_"+year))

    if cost_int_match_desc_bool == False:
        trade_ex["cost_int_match_desc"] = cost_int_match_desc
    cost_int_match_desc_bool = True

    trade_ex["bau_usd_"+year] = bau_usd
    trade_ex["levy_usd_"+year] = levy_usd
    trade_ex["asis_usd_"+year] = asis_usd
    trade_ex["brazil_usd_"+year] = brazil_usd
    trade_ex["liscr_usd_"+year] = liscr_usd
    trade_ex["direct_usd_"+year] = direct_usd
    
print("Inventory voyages table used for the following vessel types: \n\n{0}\n".format(pd.Series(vess_types_not_found_in_inventories).unique()))
print("Global voyages table used for the following vessel types found: \n\n{0}\n".format(pd.Series(vess_types_not_found_in_global_voys).unique()))
print("Global voyages table used for the following commodities as no 'facilitating vessel types' were identified: \n\n{0}\n".format(pd.Series(hs_codes_not_mapped_to_HS).unique()))

print()
print("Number of trade records present in dataset: {0}".format(trade_ex.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_ex.partnerCode.unique())))
print("\nTotal value of trade: US${0}m".format(np.round(trade_ex.fobvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_ex.vol_kg_vtw.sum() / 1e6, 1)))

print("\nDistance to market: {0} nm (min) to {1} nm (max), {2} nm (observed mean)".format(
    round(trade_ex.sea_dist_nm.min(), 1),
    round(trade_ex.sea_dist_nm.max(), 1),
    round(trade_ex.sea_dist_nm.mean(), 1)
))
print("Average distance to market (volume-weighted): {0} nm".format(
    round((trade_ex.vol_kg_vtw / 1000 * trade_ex.sea_dist_nm).sum() / (trade_ex.vol_kg_vtw.sum() / 1000), 1)
))
print("Average distance to market (value-weighted): {0} nm".format(
    round((trade_ex.fobvalue / 1000 * trade_ex.sea_dist_nm).sum() / (trade_ex.fobvalue.sum() / 1000), 1)
))

print("\n\nTotal FEM costs in 2041...\n")
print("BAU: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.bau_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.bau_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("Levy: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.levy_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.levy_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("'as-is': US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.asis_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.asis_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("Brazil: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.brazil_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.brazil_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("LISCR: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.liscr_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.liscr_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("Direct: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.direct_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.direct_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))

trade_ex_agg = trade_ex.groupby(["cost_int_match_desc"]).agg({"vol_kg_vtw":"sum", "fobvalue":"sum"}).reset_index().sort_values(by="vol_kg_vtw", ascending=False).reset_index(drop=True)
trade_ex_agg["vol_kg_vtw_pct"] = round(100 * trade_ex_agg.vol_kg_vtw / trade_ex_agg.vol_kg_vtw.sum(), 1)
trade_ex_agg["fobvalue_pct"] = round(100 * trade_ex_agg.fobvalue / trade_ex_agg.fobvalue.sum(), 1)
print("Most prominent 'cost intensity match' types...\n\n{0}\n".format(
    trade_ex_agg[["cost_int_match_desc", "vol_kg_vtw_pct", "fobvalue_pct"]].head(5)
))
trade_ex_agg_2 = trade_ex.groupby(["fac_vessel_types_str", "cost_int_match_desc"]).agg({"vol_kg_vtw":"sum", "fobvalue":"sum"}).reset_index().sort_values(by="vol_kg_vtw", ascending=False).reset_index(drop=True)
trade_ex_agg_2["vol_kg_vtw_pct"] = round(100 * trade_ex_agg_2.vol_kg_vtw / trade_ex_agg_2.vol_kg_vtw.sum(), 1)
trade_ex_agg_2["fobvalue_pct"] = round(100 * trade_ex_agg_2.fobvalue / trade_ex_agg_2.fobvalue.sum(), 1)
print("Most prominent 'facilitating vessel types' and their 'cost intensity match' types...")
display(trade_ex_agg_2[["fac_vessel_types_str", "cost_int_match_desc", "vol_kg_vtw_pct", "fobvalue_pct"]].head(5))

trade_ex_na = trade_ex[(trade_ex[["vol_kg_vtw", "primaryValue", "HS2", "HS6", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (trade_ex.fac_vessel_types.str.len() == 0)]
trade_ex = trade_ex[(~trade_ex[["vol_kg_vtw", "primaryValue", "HS2", "HS6", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (~trade_ex.fac_vessel_types.str.len() == 0)]
print("Rows with NA: ${0} ({1}%)\n".format(trade_ex_na.primaryValue.sum(), round(100 * trade_ex_na.primaryValue.sum() / trade_ex.primaryValue.sum(), 1)))

trade_ex.to_csv(output_dir + "{0}/impact_ex_{1}.csv".format(iso_country, version), index=False)
trade_ex.head(2)

Processing game year 2026
Processing game year 2029
Processing game year 2032
Processing game year 2035
Processing game year 2038
Processing game year 2041
Processing game year 2044
Processing game year 2047
Processing game year 2050
Inventory voyages table used for the following vessel types: 

['Liquefied gas tanker']

Global voyages table used for the following vessel types found: 

['Other liquids tankers']

Global voyages table used for the following commodities as no 'facilitating vessel types' were identified: 

['5703 | Carpets and other te' '5911 | Textile products and'
 '6006 | Fabrics; knitted or ' '0106 | Animals; live, n.e.c'
 '2931 | Other organo-inorgan' '3002 | Human blood; animal '
 '3004 | Medicaments; (not go' '6201 | Overcoats, car-coats'
 "6202 | Coats; women's or gi" '8517 | Telephone sets, incl'
 '8525 | Transmission apparat' '8528 | Monitors and project'
 '0306 | Crustaceans; in shel' '0307 | Molluscs; whether in'
 '3006 | Pharmaceutical goods' '3105 | Fertilize

,fac_vessel_types_str,cost_int_match_desc,vol_kg_vtw_pct,fobvalue_pct
0,['Bulk carrier'],1. Primary Int. Inv. with Vess Type,36.2,6.0
1,['Container'],1. Primary Int. Inv. with Vess Type,26.5,43.9
2,"['Container', 'Bulk carrier']",1. Primary Int. Inv. with Vess Type,25.1,44.1
3,"['Bulk carrier', 'General cargo']",1. Primary Int. Inv. with Vess Type,6.6,1.7
4,"['Bulk carrier', 'Container']",1. Primary Int. Inv. with Vess Type,1.7,0.7


Rows with NA: $0.0 (0.0%)



,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,HS6_desc,HS6,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,HS6_bilat_desc,HS2,HS2_desc,HS2_bilat_desc,HS4,HS4_desc,HS4_bilat_desc,hs2,hs4,hs6,hs6_description,eora_desc,isic,isic_desc,sitc,sitc_desc,gtap,gtap_desc,cpc,cpc_desc,usd_per_kg,kg_per_usd_cepii,_trade_ex_vtw_hs4_m,kg_per_usd_netWgt,vol_kg_vtw,vol_kg_vtw_comp,fac_vessel_types,fac_vessel_types_str,sea_dist_km,sea_dist_nm,tnm,cost_int_match_desc,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50
0,0,404,Kenya,340,Honduras,000,World,Domestic Export,Sea,"551323 | Fabrics, woven; dyed",551323,kg,5182.0,5182.0,NaN,27151.845,27151.845,"Honduras | 551323 | Fabrics, woven; dyed",55,55 | Man-made staple fibr,Honduras | 55 | Man-made staple fibr,5513,5513 | Woven fabrics of syn,Honduras | 5513 | Woven fabrics of syn,55,5513,551323,"Dyed Woven fabrics of synthetic, polyester sta...",Textiles and Wearing Apparel,1711,Preparation and spinning of textile fibres; we...,65331,"Fabrics, woven, <85% of synthetic fibres, of a...",27,TEX - Textiles,2676,"Woven fabrics of man-made staple fibres, conta...",6.4903,0.1541,both,0.190853,4184.099315,5182.0,[Container],['Container'],17795.0,11057.296945,46264.828568,1. Primary Int. Inv. with Vess Type,157.032394,156.452249,156.495416,156.495416,156.862805,156.862805,165.001574,310.572201,182.012387,167.952860,168.601499,173.432221,161.218427,295.393425,189.853130,174.698050,170.671960,179.288528,157.903496,327.427737,238.402257,207.546736,191.765614,187.373897,154.635084,347.350535,278.616337,259.978497,203.353715,201.541744,151.426736,354.596111,334.661442,340.321735,219.971883,212.004114,148.156775,348.316385,333.369825,353.937356,266.117476,216.032972,145.078741,308.376951,317.984875,332.695783,246.792073,218.674907,141.950390,277.833693,284.191791,328.595222,271.566811,220.393177
1,1,404,Kenya,826,United Kingdom,000,World,Domestic Export,Sea,"551441 | Fabrics, woven; plai",551441,kg,30.0,30.0,NaN,255.592,255.592,"United Kingdom | 551441 | Fabrics, woven; plai",55,55 | Man-made staple fibr,United Kingdom | 55 | Man-made staple fibr,5514,5514 | Woven fabrics of syn,United Kingdom | 5514 | Woven fabrics of syn,55,5514,551441,"Woven fabrics of polyester staple fibres, cont...",Textiles and Wearing Apparel,1711,Preparation and spinning of textile fibres; we...,65333,"Fabrics, woven, <85% of synthetic fibres, of a...",27,TEX - Textiles,2676,"Woven fabrics of man-made staple fibres, conta...",5.5692,0.1796,both,0.117375,45.904323,30.0,[Container],['Container'],11940.0,7419.169740,340.571966,1. Primary Int. Inv. with Vess Type,1.155972,1.151701,1.152019,1.152019,1.154723,1.154723,1.214636,2.286233,1.339858,1.236361,1.241136,1.276697,1.186787,2.174497,1.397577,1.286015,1.256377,1.319807,1.162384,2.410313,1.754964,1.527826,1.411655,1.379326,1.138324,2.556972,2.050995,1.913795,1.496959,1.483621,1.114706,2.610309,2.463563,2.505230,1.619292,1.560638,1.090635,2.564082,2.454055,2.605460,1.958986,1.590296,1.067977,2.270073,2.340800,2.449093,1.816725,1.609744,1.044948,2.045233,2.092038,2.418907,1.999101,1.622393


## 3.4 Create Results Tables

### 3.4.1 Total Costs per Scenario

In [12]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        np.round(trade_ex["bau_usd_{0}".format(year)].sum(), 2),
        np.round(trade_ex["levy_usd_{0}".format(year)].sum(), 2),
        np.round(trade_ex["asis_usd_{0}".format(year)].sum(), 2),
        np.round(trade_ex["brazil_usd_{0}".format(year)].sum(), 2),
        np.round(trade_ex["liscr_usd_{0}".format(year)].sum(), 2),
        np.round(trade_ex["direct_usd_{0}".format(year)].sum(), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


if res_mlc == True:
    print("\nTotal FEM costs by scenario (% ciffob)...\n")
    d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * trade_ex["bau_usd_{0}".format(year)].sum() / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * trade_ex["levy_usd_{0}".format(year)].sum() / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * trade_ex["asis_usd_{0}".format(year)].sum() / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * trade_ex["brazil_usd_{0}".format(year)].sum() / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * trade_ex["liscr_usd_{0}".format(year)].sum() / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * trade_ex["direct_usd_{0}".format(year)].sum() / trade_ex.TandT_ciffob.sum(), 2),
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_ex_total_costs_pct_ciffob_{1}.csv".format(iso_country, version), index=False)
    display(df)


if res_mlc == True:
    print("\nTotal FEM costs by scenario (% cif)...\n")
    d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * trade_ex["bau_usd_{0}".format(year)].sum() / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * trade_ex["levy_usd_{0}".format(year)].sum() / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * trade_ex["asis_usd_{0}".format(year)].sum() / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * trade_ex["brazil_usd_{0}".format(year)].sum() / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * trade_ex["liscr_usd_{0}".format(year)].sum() / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * trade_ex["direct_usd_{0}".format(year)].sum() / trade_ex.TandT_cifvalue.sum(), 2)
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_ex_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
    display(df)

    
print("\nTotal FEM costs by scenario (% fob)...\n")
d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_ex["bau_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 2),
        round(100 * trade_ex["levy_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 2),
        round(100 * trade_ex["asis_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 2),
        round(100 * trade_ex["brazil_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 2),
        round(100 * trade_ex["liscr_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 2),
        round(100 * trade_ex["direct_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_total_costs_pct_fob_{1}.csv".format(iso_country, version), index=False)
display(df)


print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_ex["bau_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["levy_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["asis_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["brazil_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["liscr_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["direct_usd_{0}".format(year)].sum() / gdp, 3)
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_total_costs_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,25139175.10,24666874.21,24253037.27,23845741.27,23446078.83,23038562.37,22654891.50,22264466.37
1,Levy,42946501.03,41201566.38,45357012.52,49355865.60,50253594.84,48342071.77,43772595.96,40376821.79
2,'as-is',27006746.44,28006745.72,34261522.71,39442996.03,47721426.98,46763408.57,45104079.20,41146157.25
3,Brazil,25282897.20,26333783.15,30818200.02,37185618.39,49025551.44,49440044.91,47983699.30,47062909.53
4,LISCR,25370262.20,25872590.22,28857144.78,30361349.44,33337677.98,39196393.26,37348043.96,39460042.99
5,Direct,26182522.32,26900198.34,27893701.75,29659033.85,30952633.72,31434646.01,31745557.12,31948778.67



Total FEM costs by scenario (% fob)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,0.92,0.90,0.89,0.87,0.86,0.84,0.83,0.82
1,Levy,1.57,1.51,1.66,1.81,1.84,1.77,1.61,1.48
2,'as-is',0.99,1.03,1.26,1.45,1.75,1.71,1.65,1.51
3,Brazil,0.93,0.97,1.13,1.36,1.80,1.81,1.76,1.73
4,LISCR,0.93,0.95,1.06,1.11,1.22,1.44,1.37,1.45
5,Direct,0.96,0.99,1.02,1.09,1.14,1.15,1.16,1.17



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,0.025,0.025,0.024,0.024,0.023,0.023,0.023,0.022
1,Levy,0.043,0.041,0.045,0.049,0.050,0.048,0.043,0.040
2,'as-is',0.027,0.028,0.034,0.039,0.047,0.046,0.045,0.041
3,Brazil,0.025,0.026,0.031,0.037,0.049,0.049,0.048,0.047
4,LISCR,0.025,0.026,0.029,0.030,0.033,0.039,0.037,0.039
5,Direct,0.026,0.027,0.028,0.029,0.031,0.031,0.032,0.032


### 3.4.2 Costs per Scenario relative to BAU

In [13]:
print("\nTotal FEM costs by scenario (US$)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:

    d_temp = {"20{0}".format(year): [
        np.round((trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_ex["direct_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_costs_to_bau_usd_bau_{1}.csv".format(iso_country, version), index=False)
display(df)


print("\nTotal change in FEM costs by scenario (% bau)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        round((100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_ex["direct_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_costs_to_bau_pct_bau_{1}.csv".format(iso_country, version), index=False)
display(df)


if res_mlc == True:
    print("\nTotal FEM costs by scenario (% ciffob)...\n")
    d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * (trade_ex["direct_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_ciffob.sum(), 2),
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_ex_costs_to_bau_pct_ciffob_{1}.csv".format(iso_country, version), index=False)
    display(df)


if res_mlc == True:
    print("\nTotal FEM costs by scenario (% cif)...\n")
    d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * (trade_ex["direct_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_cifvalue.sum(), 2),
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_ex_costs_to_bau_pct_cif_{1}.csv".format(iso_country, version), index=False)
    display(df)


print("\nTotal FEM costs by scenario (% fob)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 2),
        round(100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 2),
        round(100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 2),
        round(100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 2),
        round(100 * (trade_ex["direct_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_costs_to_bau_pct_fob_{1}.csv".format(iso_country, version), index=False)
display(df)


print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["direct_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_costs_to_bau_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,17807325.94,16534692.17,21103975.25,25510124.33,26807516.01,25303509.40,21117704.45,18112355.42
1,'as-is',1867571.34,3339871.50,10008485.43,15597254.76,24275348.15,23724846.20,22449187.70,18881690.88
2,Brazil,143722.11,1666908.94,6565162.75,13339877.12,25579472.60,26401482.54,25328807.80,24798443.16
3,LISCR,231087.10,1205716.01,4604107.51,6515608.17,9891599.14,16157830.89,14693152.46,17195576.62
4,Direct,1043347.22,2233324.13,3640664.47,5813292.57,7506554.88,8396083.64,9090665.61,9684312.30



Total change in FEM costs by scenario (% bau)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,70.83,67.03,87.02,106.98,114.34,109.83,93.21,81.35
1,'as-is',7.43,13.54,41.27,65.41,103.54,102.98,99.09,84.81
2,Brazil,0.57,6.76,27.07,55.94,109.10,114.60,111.80,111.38
3,LISCR,0.92,4.89,18.98,27.32,42.19,70.13,64.86,77.23
4,Direct,4.15,9.05,15.01,24.38,32.02,36.44,40.13,43.50



Total FEM costs by scenario (% fob)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,0.65,0.61,0.77,0.94,0.98,0.93,0.77,0.66
1,'as-is',0.07,0.12,0.37,0.57,0.89,0.87,0.82,0.69
2,Brazil,0.01,0.06,0.24,0.49,0.94,0.97,0.93,0.91
3,LISCR,0.01,0.04,0.17,0.24,0.36,0.59,0.54,0.63
4,Direct,0.04,0.08,0.13,0.21,0.28,0.31,0.33,0.36



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,0.018,0.016,0.021,0.025,0.027,0.025,0.021,0.018
1,'as-is',0.002,0.003,0.010,0.015,0.024,0.024,0.022,0.019
2,Brazil,0.000,0.002,0.007,0.013,0.025,0.026,0.025,0.025
3,LISCR,0.000,0.001,0.005,0.006,0.010,0.016,0.015,0.017
4,Direct,0.001,0.002,0.004,0.006,0.007,0.008,0.009,0.010
